# DarkNiche · retomar uma análise existente, sem IA e sem novas buscas
Aponte `EXISTING_RUN` para a pasta da análise no Drive (`MyDrive/DarkNiche-state/analises/<pasta>`) e envie o ZIP de código na primeira célula.
O modo dados não precisa de modelo, de Gemini API nem de crédito de assistente. Ele não avalia fit cultural nem recomenda nichos: mostra os dados coletados.
Execute tudo, escolha os recortes e idiomas e clique em Gerar relatório dos dados. Para uma análise estratégica com IA, use o notebook principal.


## 1. Enviar o pacote de código
Envie `DarkNiche-Colab.zip`. Gere-o no seu computador com `python analise-mercado-yt/extras/colab/package_colab.py` (ele cria `dist/DarkNiche-Colab.zip`) ou baixe da página de releases do projeto. Não envie `.env` nem credenciais.

In [ ]:
from google.colab import files
from pathlib import Path, PurePosixPath
import io, json, os, stat, subprocess, sys, zipfile
ROOT = Path('/content/DarkNiche')
uploads = files.upload()
bundles = [data for name, data in uploads.items() if name.lower().endswith('.zip')]
if len(bundles) != 1:
    raise ValueError('Envie exatamente um ZIP do código.')
with zipfile.ZipFile(io.BytesIO(bundles[0])) as archive:
    for item in archive.infolist():
        path = PurePosixPath(item.filename)
        if (path.is_absolute() or '..' in path.parts or '\\' in item.filename or ':' in item.filename
                or stat.S_ISLNK(item.external_attr >> 16)
                or any(p.startswith('.') for p in path.parts)):
            raise ValueError('Pacote contém caminho não permitido.')
        target = ROOT.joinpath(*path.parts)
        if not target.resolve().is_relative_to(ROOT.resolve()):
            raise ValueError('Caminho fora da pasta de código.')
    archive.extractall(ROOT)
del uploads, bundles
if not (ROOT / 'analise-mercado-yt/SKILL.md').is_file():
    raise ValueError('Pacote DarkNiche inválido.')
print('Código carregado; nenhuma chave importada.')

## 2. Dependências
A coleta básica pela API usa CPU e não precisa de ffmpeg. Transcrição e vídeo são opcionais.

In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(ROOT / 'requirements.txt'), 'ipywidgets>=8', 'openai>=1'], check=True)
print('Dependências instaladas.')

## 3. Persistência
Com Drive, retome usando a mesma pasta. Evite sessões simultâneas no mesmo contador e cache.
Sem Drive, baixe as evidências ao terminar: a VM do Colab é temporária.

In [ ]:
USE_DRIVE = True # @param {type:"boolean"}
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    STATE = Path('/content/drive/MyDrive/DarkNiche-state')
else:
    STATE = Path('/content/DarkNiche-state')
STATE.mkdir(parents=True, exist_ok=True)
os.environ['AMY_HOME'] = str(STATE / 'runtime')
os.environ.setdefault('YT_SEARCH_DAILY_BUDGET', '95')
os.environ.setdefault('YT_DAILY_BUDGET', '9500')
SKILL = ROOT / 'analise-mercado-yt'
SCRIPTS = SKILL / 'scripts'
sys.path.insert(0, str(SCRIPTS))
sys.path.insert(0, str(SKILL / 'extras/colab'))
from colab_support import migrate_legacy_state
migrated = migrate_legacy_state(Path.home() / '.analise-mercado-yt', Path(os.environ['AMY_HOME']))
if migrated:
    print('Estado local anterior preservado:', ', '.join(migrated))
from common import read_json, write_json, set_run
from selection import propose, select, require_selection, estimate_plan, require_triage
from colab_support import export_run
def command(script, *args):
    subprocess.run([sys.executable, str(SCRIPTS / script), *map(str, args)], check=True, cwd=ROOT)
print('Estado preparado. O contador é local; o uso em outras máquinas não é conhecido.')

## 4. Sem credenciais no modo dados
Este notebook lê só arquivos existentes. Não pede chaves nem consome cota do YouTube ou de IA.


In [ ]:
from google.colab import userdata
print('Modo dados: nenhuma chave do YouTube ou do Gemini é necessária para consultar os arquivos existentes.')


## 5. Criar ou retomar análise
Para retomar, informe o caminho em `EXISTING_RUN`. O pedido pode limitar idiomas e orçamento.

In [ ]:
TITLE = 'Minha análise' # @param {type:"string"}
REFERENCE = '' # @param {type:"string"}
LANGUAGES = 'pt,en,es' # @param {type:"string"}
REQUEST = 'Consultar os dados já coletados e gerar relatório técnico, sem novas buscas e sem recomendação estratégica.' # @param {type:"string"}
EXISTING_RUN = '/content/drive/MyDrive/DarkNiche-state/analises/<pasta-da-analise>' # @param {type:"string"}
if EXISTING_RUN.strip():
    if '<' in EXISTING_RUN:
        raise ValueError('Preencha EXISTING_RUN com a pasta real da análise no Drive (troque o texto entre < e >).')
    RUN = Path(EXISTING_RUN).expanduser()
    if not (RUN / 'plan.json').is_file():
        raise ValueError('Pasta existente não contém plan.json.')
else:
    from common import slug
    from datetime import date
    RUN = STATE / 'analises' / f'{slug(TITLE, 40)}-{date.today():%Y-%m}'
    if not (RUN / 'plan.json').exists():
        args = [TITLE, '--langs', LANGUAGES, '--dir', STATE / 'analises']
        if REFERENCE.strip():
            args += ['--ref', REFERENCE.strip()]
        command('init_run.py', *args)
set_run(RUN)
print('Análise:', RUN)

## 6. Referência inicial (opcional)
Só com canal ou vídeo de referência: até 60 uploads, cerca de 5 a 7 unidades gerais e zero buscas. Sem download de vídeo.
Sem referência (análise de nicho ou ideia), esta etapa não faz nada.

In [ ]:
plan = read_json(RUN / 'plan.json')
ref = plan.get('reference') or {}
COLLECT_REFERENCE = False # @param {type:"boolean"}
if COLLECT_REFERENCE and ref.get('value') in ('@handle_do_canal', '@handle', '@canal'):
    raise ValueError('A referência é um placeholder. Defina um canal real antes de analisar; ela não será trocada automaticamente.')
if COLLECT_REFERENCE and ref.get('value') and not (RUN / 'data/reference.json').exists():
    command('reference.py', '--run', RUN, ref['value'], '--max', '60', '--no-ytdlp')
print('Referência pronta; o modo ideia usa o seu pedido.')

## 7. IA e piloto automáticos
    Não envie plan.json: a IA conectada gera as hipóteses, o notebook coleta o mercado e a IA avalia as evidências.
    `dados` usa só um piloto já coletado, sem IA nem novas buscas; gera relatório técnico, sem recomendações.
    `colab` usa a IA disponível na sua sessão ou conta. Se estiver indisponível, escolha `gemini`, crie uma chave no AI Studio
    e salve `GEMINI_API_KEY` em Secrets. Para evitar cobrança, use um projeto Gemini na faixa gratuita.
    O notebook não ativa faturamento nem muda de provedor sozinho. A chave do YouTube não fornece IA.
    Os dados enviados ao modelo são o pedido e evidências públicas; não coloque informações privadas no pedido.

In [ ]:
AI_BACKEND = 'dados' # @param ["colab", "gemini", "dados"]
AI_MODEL = '' # @param {type:"string"}
PILOT_DURATION = 'longform' # @param ["medium", "long", "longform"]
PILOT_BUDGET = 24 # @param {type:"integer"}
from text_ai import TextAI, AIError
from notebook_flow import initial_analysis, finish_analysis
if AI_BACKEND == 'gemini' and not os.environ.get('GEMINI_API_KEY', '').strip():
    try:
        os.environ['GEMINI_API_KEY'] = userdata.get('GEMINI_API_KEY').strip()
    except (userdata.SecretNotFoundError, userdata.NotebookAccessError):
        raise ValueError('Crie GEMINI_API_KEY em Secrets e permita o acesso ao notebook. Use projeto gratuito para evitar cobrança.') from None
ai = TextAI(RUN, backend=AI_BACKEND, model=AI_MODEL, max_calls=8)
phase = initial_analysis(RUN, REQUEST, ai, duration=PILOT_DURATION, pilot_budget=PILOT_BUDGET)

## 8. Sua única escolha durante a análise
    Veja o ranking acima, escolha recortes e mercados, veja o custo e clique em **Aprofundar seleção**.
    Esse botão coleta os dados adicionais, escreve e valida o analysis.json e gera e baixa o relatório automaticamente.
    Se houver uma seleção válida de uma sessão anterior, o fluxo retoma essa seleção sem perguntar de novo.
    `Teto buscas/gerais` cobre matriz + sondagens sem cache; perfis e piloto são informados à parte.
    Não há upload intermediário, troca de ZIP com agente nem importação manual da análise.

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output
from selection import select
def finish_and_download():
    report = finish_analysis(RUN, REQUEST, ai)
    files.download(str(report))
    files.download(str(export_run(RUN, Path('/content/DarkNiche-analise.zip'))))
if phase == 'selected':
    finish_and_download()
else:
    proposal = read_json(RUN / 'proposal.json')
    data_only = AI_BACKEND == 'dados'
    recortes = widgets.SelectMultiple(options=[(f"{c.get('priority', '?')}. {c.get('label', c['id'])}", c['id']) for c in sorted(proposal['cells'], key=lambda c: c.get('priority', 999))], value=(), description='Recortes:')
    idiomas = widgets.SelectMultiple(options=[(f"{l['lang']}/{l.get('region', '?')}", l['lang']) for l in proposal['languages']], value=(), description='Mercados:')
    search_budget = widgets.BoundedIntText(value=60, min=0, max=95, description='Teto buscas:')
    unit_budget = widgets.BoundedIntText(value=500, min=0, max=9500, description='Teto gerais:')
    preview_button = widgets.Button(description='Ver custo da seleção')
    confirm_button = widgets.Button(description='Gerar relatório dos dados' if data_only else 'Aprofundar seleção', button_style='success')
    output = widgets.Output()
    previewed = None
    def choices():
        return (tuple(recortes.value), tuple(idiomas.value), (), search_budget.value, unit_budget.value)
    def preview(_):
        global previewed
        with output:
            clear_output()
            try:
                selected = choices()
                state = select(RUN, *selected, dry_run=True)
                print('Matriz + sondagens sem cache:', state['scan_estimate'])
                print('Resultados idênticos do piloto serão reutilizados.')
                print('Modo dados: nenhuma nova busca, perfil ou chamada de IA será executada.' if data_only else 'Perfis adicionais: até 12 concorrentes + referência, 60 uploads/canal (~5 gerais/canal), fora do teto do scan.')
                previewed = selected
            except ValueError as e:
                previewed = None
                print(e)
    def confirm(_):
        with output:
            clear_output()
            if previewed != choices():
                print('Escolha os recortes e mercados e veja o custo antes de aprofundar.')
                return
            confirm_button.disabled = True
            preview_button.disabled = True
            try:
                select(RUN, *choices())
                print('Escolha registrada. Gerando relatório dos dados existentes…' if data_only else 'Escolha registrada. Coletando, analisando e gerando o relatório…')
                finish_and_download()
                print('Concluído. O HTML e as evidências foram enviados para download.')
            except (AIError, ValueError, RuntimeError) as e:
                print(e)
                print('Dados e seleção preservados. Corrija a causa e clique de novo para retomar.')
            finally:
                confirm_button.disabled = False
                preview_button.disabled = False
    preview_button.on_click(preview)
    confirm_button.on_click(confirm)
    display(widgets.VBox([recortes, idiomas, search_budget, unit_budget, preview_button, confirm_button, output]))

**Executar tudo:** preencha os campos antes e use Ambiente de execução → Executar tudo.
    Na primeira preparação haverá upload do código e, se você ligou o Drive, o login nele. Depois, plano, piloto e ranking
    são automáticos; o notebook termina mostrando a seleção. O botão **Aprofundar seleção** segue até o HTML.
    Se a IA, a cota ou a rede falharem, aparece um erro explícito e o estado fica salvo; não há seletor de arquivo oculto.
    Guia: `analise-mercado-yt/extras/colab/colab.md`. A faixa gratuita do Gemini depende do projeto e da conta; não é garantida
    pela chave nem por usar o Colab gratuito. [Preços oficiais](https://ai.google.dev/gemini-api/docs/pricing).